# 01 — Data Preparation

Đọc trực tiếp Excel raw; chỉ chuẩn hóa kiểu dữ liệu, tạo trường phân tích và cờ chất lượng. Giữ nguyên mọi record và missing. Không xác nhận grain hoặc công thức rate khi nguồn chưa cung cấp.

Rule: không fill/drop, deduplicate theo Post_ID, cắt outlier hoặc tính lại rate. Giá trị ngoài miền hợp lý được gắn cờ; record đó chỉ bị loại khỏi phép tính KPI liên quan, không bị xóa khỏi file. Bảng rule và kết quả được lưu trong `04_data_quality_summary.md`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/Social_Media_Content_Performance_Dataset.xlsx").exists())
out = next(p for p in root.iterdir() if p.is_dir() and p.name.startswith("2."))
artifacts = out / "artifacts"
tables = artifacts / "tables"
figures = artifacts / "figures"
for folder in [artifacts, tables, figures]:
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)


In [2]:
raw_path = root / "data/Social_Media_Content_Performance_Dataset.xlsx"
raw = pd.read_excel(raw_path, sheet_name="Sheet1")
df = raw.copy()
df.insert(0, "Source_Row", np.arange(2, len(df) + 2))
counts = ["Engagement", "Views", "Likes", "Shares", "Comments", "Impressions",
          "Video_Views", "Live_Stream_Views", "Clicks"]
rates = ["Engagement_Rate", "Click_Through_Rate"]
dates = ["Post_Published_At", "Post_Date"]
dimensions = ["Post_ID", "Platform", "Content_Type", "Content_Category", "Post_Type",
              "Region", "Main_Hashtag", "Engagement_Level"]
conversion_failures = {}
for col in counts + rates + ["Longitude", "Latitude", "Post_Hour"]:
    converted = pd.to_numeric(df[col], errors="coerce")
    conversion_failures[col] = int((df[col].notna() & converted.isna()).sum())
    df[col] = converted
for col in dates:
    converted = pd.to_datetime(df[col], errors="coerce")
    conversion_failures[col] = int((df[col].notna() & converted.isna()).sum())
    df[col] = converted
for col in dimensions:
    df[col] = df[col].astype("string")  # no label rewriting or null filling
assert not any(conversion_failures.values()), conversion_failures
df["Post_Hour"] = df["Post_Hour"].astype("Int64")
for col in counts:
    df[col] = df[col].astype("Float64")  # preserve nullable counts
df["Month"] = df["Post_Date"].dt.strftime("%Y-%m")
df["Weekday"] = df["Post_Date"].dt.day_name()
last_month = df["Post_Date"].max().to_period("M")
df["Partial_Month"] = df["Post_Date"].dt.to_period("M").eq(last_month) & (
    df["Post_Date"].max().day < df["Post_Date"].max().days_in_month)
display(df.dtypes.rename("analysis_type").to_frame())


,analysis_type
Source_Row,int64
Post_ID,string[python]
Platform,string[python]
Content_Type,string[python]
Content_Category,string[python]
Post_Type,string[python]
Region,string[python]
Longitude,float64
Latitude,float64
Engagement,Float64


In [3]:
quality = []
for col in counts:
    bad = df[col].notna() & ((df[col] < 0) | ~np.isfinite(df[col].astype(float)) |
                             (df[col] % 1 != 0))
    df["Invalid_" + col] = bad.fillna(False).astype(bool)
    quality.append([col, "nonnegative finite integer count", int(bad.sum())])
for col in rates:
    bad = df[col].notna() & ((df[col] < 0) | (df[col] > 1) |
                             ~np.isfinite(df[col].astype(float)))
    df["Invalid_" + col] = bad.fillna(False).astype(bool)
    quality.append([col, "finite decimal within [0, 1]; screening convention", int(bad.sum())])
df["Repeated_Post_ID"] = df.duplicated("Post_ID", keep=False)
candidate_key = ["Post_ID", "Platform", "Post_Published_At"]
df["Repeated_Candidate_Key"] = df.duplicated(candidate_key, keep=False)
quality.extend([
    ["all raw columns", "exact duplicate beyond first", int(raw.duplicated().sum())],
    ["Post_ID", "duplicate beyond first", int(raw.duplicated("Post_ID").sum())],
    ["candidate key", "duplicate beyond first", int(raw.duplicated(candidate_key).sum())],
    ["Post_Date", "timestamp/date mismatch", int(df.Post_Published_At.dt.normalize().ne(df.Post_Date).sum())],
    ["Post_Hour", "timestamp/hour mismatch", int(df.Post_Published_At.dt.hour.ne(df.Post_Hour).sum())],
    ["Post_Hour", "outside [0, 23]", int((~df.Post_Hour.between(0, 23)).sum())],
    ["Longitude", "outside [-180, 180]", int((~df.Longitude.between(-180, 180)).sum())],
    ["Latitude", "outside [-90, 90]", int((~df.Latitude.between(-90, 90)).sum())],
    ["dimensions", "whitespace-only label", sum(int(df[c].str.strip().eq("").sum()) for c in dimensions)],
    ["columns", "duplicate column names", int(df.columns.duplicated().sum())]
])
quality_table = pd.DataFrame(quality, columns=["field", "check", "records"])
missing = pd.DataFrame({"missing_count": raw.isna().sum(),
                        "missing_pct": raw.isna().mean().mul(100)})
display(quality_table)
display(missing)
print("Clicks/CTR missing masks match:", raw.Clicks.isna().equals(raw.Click_Through_Rate.isna()))
print("Engagement != Likes + Shares + Comments:", int(raw.Engagement.ne(raw.Likes+raw.Shares+raw.Comments).sum()))
print("Video_Views > Views:", int(raw.Video_Views.gt(raw.Views).sum()))
print("Live_Stream_Views > Views:", int(raw.Live_Stream_Views.gt(raw.Views).sum()))
print("Post_ID with more than one category:", int((raw.groupby("Post_ID").Content_Category.nunique() > 1).sum()))
print("Maximum difference from candidate ER formula:", (raw.Engagement_Rate - raw.Engagement/raw.Views).abs().max())
print("Maximum difference from candidate CTR formula:", (raw.Click_Through_Rate - raw.Clicks/raw.Impressions).abs().max())
print("These cross-metric differences are unresolved definitions, not automatic invalidation.")
quality_table.to_csv(tables / "data_quality_checks.csv", index=False)
missing.to_csv(tables / "raw_missing_values.csv", index_label="field")


,field,check,records
0,Engagement,nonnegative finite integer count,0
1,Views,nonnegative finite integer count,0
2,Likes,nonnegative finite integer count,0
3,Shares,nonnegative finite integer count,0
4,Comments,nonnegative finite integer count,0
5,Impressions,nonnegative finite integer count,0
6,Video_Views,nonnegative finite integer count,0
7,Live_Stream_Views,nonnegative finite integer count,0
8,Clicks,nonnegative finite integer count,0
9,Engagement_Rate,"finite decimal within [0, 1]; screening convention",0


,missing_count,missing_pct
Post_ID,0,0.000000
Platform,0,0.000000
Content_Type,0,0.000000
Content_Category,0,0.000000
Post_Type,0,0.000000
Region,0,0.000000
Longitude,0,0.000000
Latitude,0,0.000000
Engagement,0,0.000000
Views,0,0.000000


Clicks/CTR missing masks match: True
Engagement != Likes + Shares + Comments: 5600
Video_Views > Views: 2
Live_Stream_Views > Views: 513
Post_ID with more than one category: 153
Maximum difference from candidate ER formula: 0.0009581404581550101
Maximum difference from candidate CTR formula: 0.0008448275862068966
These cross-metric differences are unresolved definitions, not automatic invalidation.


In [4]:
analysis_path = artifacts / "analysis_ready.csv"
df.to_csv(analysis_path, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
check = pd.read_csv(analysis_path, parse_dates=dates)
assert len(check) == len(raw)
assert check.Source_Row.is_unique
for col in counts + rates:
    assert check[col].isna().equals(raw[col].isna())
    assert np.allclose(check[col].dropna().astype(float), raw[col].dropna().astype(float))
rules = """# Data quality summary

Nguồn: Excel raw, sheet Sheet1. File analysis-ready là CSV UTF-8 riêng; không ghi đè raw. Đơn vị phân tích là record nguồn, không phải unique post đã xác nhận.

| Hạng mục | Rule xử lý |
| --- | --- |
| Data type | Parse date/numeric; dừng nếu chuyển kiểu làm mất giá trị. String giữ nguyên label; count nullable. Khi đọc CSV, parse lại date theo notebook. |
| Missing | Giữ nguyên null; không fill 0. Click/CTR chỉ tính trên record hợp lệ có dữ liệu; luôn kèm coverage. |
| Duplicate | Giữ nguyên record và thêm cờ mã/khóa lặp; chưa có cơ sở chọn record đại diện. Source_Row định vị dòng Excel, không thay cho business key. |
| KPI bất hợp lý | Gắn cờ count âm, không hữu hạn, không nguyên và rate ngoài [0,1]. Chỉ mask KPI tương ứng trong phân tích; không xóa dòng. Miền rate là rule kiểm tra, không xác nhận công thức. |
| Quan hệ KPI | Không sửa Engagement theo tổng Likes/Shares/Comments hoặc tính lại rate. Chênh lệch giữa các chỉ số được ghi nhận là định nghĩa chưa rõ. |
| Outlier | Không winsorize/cắt/drop; dùng cả trung bình và median để đọc phân phối. |
| Format | Tên cột và label nguồn giữ nguyên; thêm Month, Weekday, Partial_Month và các cờ kiểm tra. |
| Time | Tháng cuối chỉ đến 01/05/2025 được đánh dấu partial; không so total tháng này với tháng trọn. Performance gắn với ngày đăng, không phải ngày phát sinh tương tác. |

"""
rules += f"Raw: **{len(raw):,} dòng, {raw.shape[1]} cột**; analysis-ready: **{len(df):,} dòng, {df.shape[1]} cột**. Số dòng bị xóa: **0**; missing bị fill: **0**; số lỗi chuyển kiểu: **0**.\n\n"
rules += quality_table.to_markdown(index=False) + "\n\n"
rules += "Missing ở Clicks và Click_Through_Rate: 3.740 mỗi cột (66,79%); các cột raw còn lại không có null.\n\n"
rules += "Grain, thời điểm trích xuất, cửa sổ đo KPI, timezone và công thức rate chưa xác nhận. Tổng absolute là tổng record trong file, chưa thể coi là lượng duy nhất của doanh nghiệp. Rate dùng mean/median nguồn ở cấp record, không phải rate tổng hợp có mẫu số xác nhận.\n\n"
rules += "Các mã bài có thể khác category giữa record; không chọn category đầu tiên để đại diện. Kiểm tra độ nhạy dùng trọng số 1 / số record của Post_ID và vẫn giữ category của từng record.\n"
(out / "04_data_quality_summary.md").write_text(rules, encoding="utf-8")
print("Saved:", analysis_path.relative_to(root), "Rows:", len(df), "Columns:", df.shape[1])


Saved: 2. Xác định vấn đề\artifacts\analysis_ready.csv Rows: 5600 Columns: 41
